# Notebook 2: Veri Seti İstatistikleri + v5 Zamanlama + Hata Analizi + Perceptual Aliasing

Bu sürüm dataset istatistiklerini korur; model yükleme, timing, hata analizi ve aliasing bölümleri v5 ViT-B/CLS artifact'leriyle çalışır.

In [ ]:
# ============================================================
# Hücre 1: Kütüphane Kurulumu ve Veri Yükleme
# ============================================================
!pip install faiss-cpu folium -q

import os, json, random, zipfile, shutil, math, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
from collections import defaultdict, Counter
from PIL import Image, ImageOps
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import faiss

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DRIVE_ZIP     = "/content/drive/My Drive/kirsehir_data.zip"
LOCAL_ROOT    = "/content/kirsehir_data"
OUTPUT_DIR    = "/content/drive/My Drive/Kirsehir_VPR_DINOv2_v5_Final"
RESULTS_DIR   = "/content/drive/My Drive/Kirsehir_VPR_Results"

MODEL_NAME    = "dinov2_vitb14"
MODEL_POOLING = "cls"
MODEL_CHECKPOINT_NAME = "dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth"
DRIVE_MODEL_PATH = os.path.join(OUTPUT_DIR, MODEL_CHECKPOINT_NAME)

OUTFEATURES   = 512
IMG_SIZE      = 518
BATCH_SIZE    = 128
NUM_WORKERS   = 4
TOP_K         = 20
GRID_SIZE     = 0.002

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(LOCAL_ROOT):
    shutil.copy2(DRIVE_ZIP, "/content/kirsehir_data.zip")
    with zipfile.ZipFile("/content/kirsehir_data.zip", 'r') as zf:
        zf.extractall(LOCAL_ROOT)
    os.remove("/content/kirsehir_data.zip")

entries = os.listdir(LOCAL_ROOT)
if len(entries) == 1 and os.path.isdir(os.path.join(LOCAL_ROOT, entries[0])):
    LOCAL_ROOT = os.path.join(LOCAL_ROOT, entries[0])

def extract_lat_lon_heading(filepath):
    parts = os.path.splitext(os.path.basename(filepath))[0].split("_")
    try:
        return float(parts[0]), float(parts[1]), float(parts[2].replace("h", "") if len(parts)>2 else "0")
    except: return None, None, None

all_images = []
for street_name in sorted(os.listdir(LOCAL_ROOT)):
    street_path = os.path.join(LOCAL_ROOT, street_name)
    if not os.path.isdir(street_path) or street_name == "model": continue
    for fname in sorted(os.listdir(street_path)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")): continue
        fpath = os.path.join(street_path, fname)
        lat, lon, heading = extract_lat_lon_heading(fpath)
        if lat is not None:
            all_images.append({
                "filepath": fpath, "street": street_name,
                "lat": lat, "lon": lon, "heading": heading,
                "point_id": f"{lat:.6f}_{lon:.6f}",
                "block_id": f"{int(lat/GRID_SIZE)}_{int(lon/GRID_SIZE)}"
            })

df_all = pd.DataFrame(all_images).sort_values(by=["point_id", "heading"])

unique_blocks = df_all["block_id"].unique().tolist()
random.shuffle(unique_blocks)
n_blocks = len(unique_blocks)
train_blocks = set(unique_blocks[:int(n_blocks * 0.70)])
test_blocks  = set(unique_blocks[int(n_blocks * 0.70):])

df_train = df_all[df_all["block_id"].isin(train_blocks)].copy()
df_test = df_all[df_all["block_id"].isin(test_blocks)].copy()
point_counts = df_test['point_id'].value_counts()
valid_points = point_counts[point_counts >= 2].index
df_test = df_test[df_test['point_id'].isin(valid_points)]
df_query = df_test.groupby("point_id").tail(1).reset_index(drop=True)
df_db = df_test[~df_test["filepath"].isin(set(df_query["filepath"]))].reset_index(drop=True)

print("✅ Veri yüklendi.")
print(f"✅ v5 model path: {DRIVE_MODEL_PATH}")

In [ ]:
# ============================================================
# Hücre 2: VERİ SETİ İSTATİSTİKLERİ (Makale Bölüm 3/4 için)
# ============================================================

print(f"{'='*60}")
print(f"📊 KIRŞEHİR VPR VERİ SETİ İSTATİSTİKLERİ")
print(f"{'='*60}")

# Genel istatistikler
n_total_images = len(df_all)
n_unique_points = df_all["point_id"].nunique()
n_streets = df_all["street"].nunique()
n_blocks_total = df_all["block_id"].nunique()

print(f"\n── Genel ──")
print(f"   Toplam görüntü sayısı       : {n_total_images:,}")
print(f"   Benzersiz koordinat noktası  : {n_unique_points:,}")
print(f"   Benzersiz sokak/cadde sayısı : {n_streets}")
print(f"   Grid blok sayısı             : {n_blocks_total}")
print(f"   Grid boyutu                  : {GRID_SIZE}° (≈{GRID_SIZE * 111000:.0f}m)")

# Nokta başına görüntü dağılımı
imgs_per_point = df_all.groupby("point_id").size()
print(f"\n── Nokta Başına Görüntü ──")
print(f"   Ortalama: {imgs_per_point.mean():.1f}")
print(f"   Min/Max : {imgs_per_point.min()} / {imgs_per_point.max()}")
print(f"   Mod     : {imgs_per_point.mode().values[0]}")

# Heading dağılımı
heading_counts = df_all["heading"].value_counts().sort_index()
print(f"\n── Yön (Heading) Dağılımı ──")
for h, c in heading_counts.items():
    direction = {0: "Kuzey", 90: "Doğu", 180: "Güney", 270: "Batı"}.get(h, f"{h}°")
    print(f"   {direction} ({h}°): {c:,} görüntü")

# Train/Test split detayları
print(f"\n── Train/Test Bölünmesi ──")
print(f"   Train blok: {len(train_blocks)} ({len(train_blocks)/n_blocks_total*100:.0f}%)")
print(f"   Test blok : {len(test_blocks)} ({len(test_blocks)/n_blocks_total*100:.0f}%)")
print(f"   Train görüntü : {len(df_train):,}")
print(f"   Test - DB      : {len(df_db):,}")
print(f"   Test - Query   : {len(df_query):,}")

# Sokak başına görüntü dağılımı
imgs_per_street = df_all.groupby("street").size().sort_values(ascending=False)
print(f"\n── Sokak Başına Görüntü (Top-10) ──")
for street, count in imgs_per_street.head(10).items():
    print(f"   {street}: {count}")

# GPS kapsam alanı
lat_range = (df_all["lat"].min(), df_all["lat"].max())
lon_range = (df_all["lon"].min(), df_all["lon"].max())

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2, dp, dl = map(math.radians, [lat1, lat2, lat2-lat1, lon2-lon1])
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

ns_dist = haversine(lat_range[0], lon_range[0], lat_range[1], lon_range[0])
ew_dist = haversine(lat_range[0], lon_range[0], lat_range[0], lon_range[1])

print(f"\n── Coğrafi Kapsam ──")
print(f"   Enlem aralığı : {lat_range[0]:.6f} – {lat_range[1]:.6f}")
print(f"   Boylam aralığı: {lon_range[0]:.6f} – {lon_range[1]:.6f}")
print(f"   K-G mesafe     : {ns_dist:.0f}m ({ns_dist/1000:.2f}km)")
print(f"   D-B mesafe     : {ew_dist:.0f}m ({ew_dist/1000:.2f}km)")
print(f"   Yaklaşık alan  : {ns_dist * ew_dist / 1e6:.2f} km²")

# Görüntü boyutu istatistikleri (ilk 100 görüntüden örnekle)
sample_sizes = []
for fp in df_all["filepath"].values[:100]:
    try:
        img = Image.open(fp)
        sample_sizes.append(img.size)  # (width, height)
    except: pass

if sample_sizes:
    widths, heights = zip(*sample_sizes)
    print(f"\n── Görüntü Boyutu (İlk 100 örnekten) ──")
    print(f"   Genişlik: {min(widths)}–{max(widths)} px (mod: {Counter(widths).most_common(1)[0][0]})")
    print(f"   Yükseklik: {min(heights)}–{max(heights)} px (mod: {Counter(heights).most_common(1)[0][0]})")


In [ ]:
# ============================================================
# Hücre 3: Veri Seti Harita Görselleştirmesi (Makale Figürü)
# ============================================================
import folium
from folium.plugins import HeatMap

# Train/Test bölgelerini gösteren harita
center_lat = df_all["lat"].mean()
center_lon = df_all["lon"].mean()

# Matplotlib ile scatter plot (makale için daha uygun)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Sol: Train vs Test bölgeleri
train_pts = df_train.drop_duplicates("point_id")
test_pts = df_test.drop_duplicates("point_id")

axes[0].scatter(train_pts["lon"], train_pts["lat"], c="blue", s=3, alpha=0.5, label=f"Train ({len(train_pts)} pts)")
axes[0].scatter(test_pts["lon"], test_pts["lat"], c="red", s=3, alpha=0.5, label=f"Test ({len(test_pts)} pts)")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
axes[0].set_title("Geographic Train/Test Split (Grid-based)", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Sağ: Sokak başına yoğunluk
street_counts = df_all.groupby("street").agg(
    lat=("lat", "mean"), lon=("lon", "mean"), count=("filepath", "count")
).reset_index()

scatter = axes[1].scatter(street_counts["lon"], street_counts["lat"],
                          c=street_counts["count"], s=street_counts["count"]*0.5,
                          cmap="YlOrRd", alpha=0.7)
axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude")
axes[1].set_title("Image Density per Street", fontsize=13)
plt.colorbar(scatter, ax=axes[1], label="Image Count")
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "dataset_map.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {fig_path}")

# İstatistik tablosu CSV
stats = {
    "Total Images": n_total_images,
    "Unique Coordinate Points": n_unique_points,
    "Unique Streets": n_streets,
    "Grid Blocks": n_blocks_total,
    "Grid Size (degrees)": GRID_SIZE,
    "Train Images": len(df_train),
    "DB Images": len(df_db),
    "Query Images": len(df_query),
    "Coverage N-S (m)": round(ns_dist),
    "Coverage E-W (m)": round(ew_dist),
    "Area (km2)": round(ns_dist * ew_dist / 1e6, 2),
}
pd.DataFrame([stats]).T.to_csv(os.path.join(RESULTS_DIR, "dataset_statistics.csv"), header=["Value"])
print(f"✅ İstatistikler kaydedildi.")


In [ ]:
# ============================================================
# Hücre 4: v5 Model Yükleme ve Değerlendirme (Hata Analizi İçin)
# ============================================================

def open_rgb_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=1).pow(1.0 / self.p)

class VPRDINOv2(nn.Module):
    def __init__(self, backbone_name=MODEL_NAME, out_dim=OUTFEATURES, pooling=MODEL_POOLING):
        super().__init__()
        self.pooling = pooling.lower()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, verbose=False)
        if self.pooling == "gem":
            self.gem = GeMPooling(p=3)
        elif self.pooling != "cls":
            raise ValueError(f"Desteklenmeyen pooling: {pooling}")
        embed_dim = self.backbone.embed_dim
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.10),
            nn.Linear(embed_dim, out_dim),
        )
    def forward(self, x):
        ret = self.backbone.forward_features(x)
        if self.pooling == "cls":
            x = ret["x_norm_clstoken"]
        else:
            x = self.gem(ret["x_norm_patchtokens"])
        x = self.head(x)
        return nn.functional.normalize(x, p=2, dim=1)

class StandardVPRDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return self.transform(open_rgb_image(self.df.iloc[idx]["filepath"])), idx

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if not os.path.exists(DRIVE_MODEL_PATH):
    raise FileNotFoundError(
        f"v5 checkpoint bulunamadı:\n{DRIVE_MODEL_PATH}\n"
        "Önce v5 ana eğitim notebook'unu çalıştır."
    )

model = VPRDINOv2(MODEL_NAME, OUTFEATURES, MODEL_POOLING).to(DEVICE)
state = torch.load(DRIVE_MODEL_PATH, map_location=DEVICE)
if isinstance(state, dict) and "model_state" in state:
    state = state["model_state"]
model.load_state_dict(state, strict=True)
model.eval()

@torch.no_grad()
def extract_embeddings(model, df, desc="Emb"):
    loader = DataLoader(StandardVPRDataset(df, eval_transform), batch_size=BATCH_SIZE,
                        num_workers=NUM_WORKERS, pin_memory=True)
    embs = []
    for imgs, _ in tqdm(loader, desc=desc):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            embs.append(model(imgs).detach().cpu().numpy())
    return np.concatenate(embs, axis=0)

print("⏳ v5 embedding çıkarımı (zamanlama ile)...")

single_times = []
for i in range(min(100, len(df_query))):
    tensor = eval_transform(open_rgb_image(df_query.iloc[i]["filepath"])).unsqueeze(0).to(DEVICE)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time()
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        _ = model(tensor)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    single_times.append(time.time() - t0)

t0 = time.time()
db_embeddings = extract_embeddings(model, df_db, "DB Emb v5")
t_db_total = time.time() - t0

t0 = time.time()
query_embeddings = extract_embeddings(model, df_query, "Query Emb v5")
t_query_total = time.time() - t0

db_emb_n = db_embeddings.copy().astype(np.float32)
qry_emb_n = query_embeddings.copy().astype(np.float32)
faiss.normalize_L2(db_emb_n)
faiss.normalize_L2(qry_emb_n)

index = faiss.IndexFlatIP(OUTFEATURES)
index.add(db_emb_n)

t0 = time.time()
dists_all, idxs_all = index.search(qry_emb_n, TOP_K)
t_faiss = time.time() - t0

db_meta = df_db[["filepath","street","lat","lon","heading","point_id"]].to_dict("records")

print(f"\n✅ v5 model ve embedding'ler hazır.")

In [ ]:
# ============================================================
# Hücre 5: HESAPLAMA MALİYETİ VE ÇALIŞMA ZAMANI ANALİZİ
# ============================================================

print(f"{'='*60}")
print(f"⏱️ HESAPLAMA MALİYETİ ANALİZİ")
print(f"{'='*60}")

print(f"\n── Donanım Bilgisi ──")
print(f"   Cihaz: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU  : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\n── DINOv2 Embedding Çıkarım Süreleri ──")
print(f"   Tek görüntü (ortalama) : {np.mean(single_times)*1000:.1f} ms")
print(f"   Tek görüntü (medyan)   : {np.median(single_times)*1000:.1f} ms")
print(f"   DB seti ({len(df_db)} görüntü)  : {t_db_total:.1f} s")
print(f"   Query seti ({len(df_query)} görüntü): {t_query_total:.1f} s")

print(f"\n── FAISS Arama Süresi ──")
print(f"   Tüm sorgular (Top-{TOP_K}) : {t_faiss*1000:.1f} ms")
print(f"   Sorgu başına             : {t_faiss/len(df_query)*1000:.3f} ms")

# Model parametre sayısı
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n── Model Boyutu ──")
print(f"   Toplam parametre    : {total_params:,}")
print(f"   Eğitilebilir param. : {trainable_params:,}")
print(f"   Embedding boyutu   : {OUTFEATURES}")
print(f"   Giriş boyutu       : {IMG_SIZE}x{IMG_SIZE}")

# Toplam pipeline süresi (tek sorgu)
t_pipeline_single = np.median(single_times) + t_faiss/len(df_query)
print(f"\n── Toplam Pipeline Süresi (Tek Sorgu) ──")
print(f"   Aşama 1 (DINOv2 + FAISS) : {t_pipeline_single*1000:.1f} ms")
print(f"   Aşama 2 (LightGlue x20)  : Notebook 1'den alınacak")
print(f"   NOT: LightGlue süresini Notebook 1'deki lg_timings'den ekleyin")

# Tablo olarak kaydet
timing_data = {
    "Operation": [
        "Single image embedding (DINOv2)",
        f"DB embedding ({len(df_db)} images)",
        f"Query embedding ({len(df_query)} images)",
        f"FAISS search (Top-{TOP_K}, all queries)",
        "FAISS search (per query)",
        "Stage 1 total (per query)",
    ],
    "Time": [
        f"{np.mean(single_times)*1000:.1f} ms",
        f"{t_db_total:.1f} s",
        f"{t_query_total:.1f} s",
        f"{t_faiss*1000:.1f} ms",
        f"{t_faiss/len(df_query)*1000:.3f} ms",
        f"{t_pipeline_single*1000:.1f} ms",
    ]
}
timing_df = pd.DataFrame(timing_data)
timing_df.to_csv(os.path.join(RESULTS_DIR, "v5_timing_analysis.csv"), index=False)
print(f"\n✅ Zamanlama tablosu kaydedildi.")

In [ ]:
# ============================================================
# Hücre 6: HATA ANALİZİ (Başarısız Durumların İncelenmesi)
# ============================================================

# Her sorgu için hata hesapla
errors = []
for i in range(len(df_query)):
    q = df_query.iloc[i]
    best = db_meta[idxs_all[i, 0]]
    err = haversine(q["lat"], q["lon"], best["lat"], best["lon"])
    errors.append({
        "query_idx": i,
        "query_street": q["street"],
        "query_lat": q["lat"],
        "query_lon": q["lon"],
        "pred_street": best["street"],
        "pred_lat": best["lat"],
        "pred_lon": best["lon"],
        "error_m": err,
        "correct_street": q["street"] == best["street"],
        "dino_score": float(dists_all[i, 0])
    })

df_errors = pd.DataFrame(errors)

print(f"{'='*60}")
print(f"🔍 HATA ANALİZİ")
print(f"{'='*60}")

# Hata kategorileri
bins = [0, 25, 50, 100, 250, 500, float('inf')]
labels_bin = ["<25m", "25-50m", "50-100m", "100-250m", "250-500m", ">500m"]
df_errors["error_bin"] = pd.cut(df_errors["error_m"], bins=bins, labels=labels_bin)

print(f"\n── Hata Dağılımı ──")
for cat in labels_bin:
    count = (df_errors["error_bin"] == cat).sum()
    pct = count / len(df_errors) * 100
    print(f"   {cat:>10}: {count:>4} sorgu ({pct:.1f}%)")

# En kötü performans gösteren sokaklar
print(f"\n── En Yüksek Hatalı Sokaklar (Medyan Hata > 200m) ──")
street_errors = df_errors.groupby("query_street").agg(
    median_error=("error_m", "median"),
    mean_error=("error_m", "mean"),
    max_error=("error_m", "max"),
    count=("error_m", "count"),
    correct_pct=("correct_street", "mean")
).sort_values("median_error", ascending=False)

high_error_streets = street_errors[street_errors["median_error"] > 200]
if len(high_error_streets) > 0:
    for street, row in high_error_streets.head(10).iterrows():
        print(f"   {street}: med={row['median_error']:.0f}m, max={row['max_error']:.0f}m, "
              f"n={row['count']:.0f}, doğru={row['correct_pct']*100:.0f}%")
else:
    print("   200m üzeri medyan hatası olan sokak yok.")

# En iyi performans gösteren sokaklar
print(f"\n── En Düşük Hatalı Sokaklar (Medyan Hata < 10m) ──")
low_error_streets = street_errors[street_errors["median_error"] < 10]
for street, row in low_error_streets.head(10).iterrows():
    print(f"   {street}: med={row['median_error']:.0f}m, n={row['count']:.0f}, "
          f"doğru={row['correct_pct']*100:.0f}%")

# Yanlış sokak tahminlerinin analizi
wrong_street = df_errors[~df_errors["correct_street"]]
print(f"\n── Yanlış Sokak Tahmini ──")
print(f"   Toplam: {len(wrong_street)} / {len(df_errors)} ({len(wrong_street)/len(df_errors)*100:.1f}%)")

# En sık karıştırılan sokak çiftleri
if len(wrong_street) > 0:
    confusion_pairs = wrong_street.groupby(["query_street", "pred_street"]).size().sort_values(ascending=False)
    print(f"\n── En Sık Karıştırılan Sokak Çiftleri (Top-10) ──")
    for (q_st, p_st), count in confusion_pairs.head(10).items():
        print(f"   {q_st} → {p_st}: {count} kez")


In [ ]:
# ============================================================
# Hücre 7: Hata Analizi Görselleştirmesi (Makale Figürleri)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Sol üst: Hata histogramı
axes[0,0].hist(df_errors["error_m"], bins=50, range=(0, 1000), color="#6366f1", edgecolor="white", alpha=0.8)
axes[0,0].axvline(x=df_errors["error_m"].median(), color="red", linestyle="--",
                  label=f"Median: {df_errors['error_m'].median():.0f}m")
axes[0,0].set_xlabel("Localization Error (m)")
axes[0,0].set_ylabel("Number of Queries")
axes[0,0].set_title("Error Distribution")
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)

# Sağ üst: Doğru/Yanlış sokak tahmini scatter
correct = df_errors[df_errors["correct_street"]]
wrong = df_errors[~df_errors["correct_street"]]
axes[0,1].scatter(correct["query_lon"], correct["query_lat"], c="green", s=5, alpha=0.5, label=f"Correct ({len(correct)})")
axes[0,1].scatter(wrong["query_lon"], wrong["query_lat"], c="red", s=8, alpha=0.7, label=f"Wrong ({len(wrong)})")
axes[0,1].set_xlabel("Longitude"); axes[0,1].set_ylabel("Latitude")
axes[0,1].set_title("Spatial Distribution of Errors")
axes[0,1].legend(fontsize=9)
axes[0,1].set_aspect('equal')
axes[0,1].grid(alpha=0.3)

# Sol alt: DINO skoru vs hata ilişkisi
axes[1,0].scatter(df_errors["dino_score"], df_errors["error_m"], s=5, alpha=0.3, c="blue")
axes[1,0].set_xlabel("DINOv2 Cosine Similarity Score")
axes[1,0].set_ylabel("Localization Error (m)")
axes[1,0].set_title("Retrieval Confidence vs Error")
axes[1,0].set_ylim(0, 2000)
axes[1,0].grid(alpha=0.3)

# Sağ alt: Sokak başına hata box plot (en çok sorgusu olan 15 sokak)
top_streets = df_errors["query_street"].value_counts().head(15).index.tolist()
df_top = df_errors[df_errors["query_street"].isin(top_streets)]
street_order = df_top.groupby("query_street")["error_m"].median().sort_values().index

box_data = [df_top[df_top["query_street"] == s]["error_m"].values for s in street_order]
bp = axes[1,1].boxplot(box_data, vert=True, patch_artist=True)
axes[1,1].set_xticklabels([s[:12] for s in street_order], rotation=45, ha="right", fontsize=8)
axes[1,1].set_ylabel("Error (m)")
axes[1,1].set_title("Per-Street Error Distribution (Top-15 by query count)")
axes[1,1].set_ylim(0, 1000)
axes[1,1].grid(alpha=0.3, axis='y')
for patch in bp['boxes']:
    patch.set_facecolor('#93c5fd')

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "v5_error_analysis.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {fig_path}")

# Hata analizi tablosu kaydet
df_errors.to_csv(os.path.join(RESULTS_DIR, "v5_per_query_errors.csv"), index=False)
street_errors.to_csv(os.path.join(RESULTS_DIR, "v5_per_street_errors.csv"))
print(f"✅ Detaylı hata tabloları kaydedildi.")


In [ ]:
# ============================================================
# Hücre 8: EN KÖTÜ 10 TAHMİN (Görsel İnceleme)
# ============================================================

worst_10 = df_errors.nlargest(10, "error_m")

fig, axes = plt.subplots(10, 2, figsize=(12, 40))

for row_idx, (_, row) in enumerate(worst_10.iterrows()):
    qi = row["query_idx"]

    # Sorgu görüntüsü
    q_img = Image.open(df_query.iloc[qi]["filepath"])
    axes[row_idx, 0].imshow(q_img)
    axes[row_idx, 0].set_title(f"QUERY: {row['query_street'][:25]}", fontsize=9, color="blue")
    axes[row_idx, 0].axis("off")

    # En iyi eşleşme
    best = db_meta[idxs_all[qi, 0]]
    p_img = Image.open(best["filepath"])
    axes[row_idx, 1].imshow(p_img)
    axes[row_idx, 1].set_title(f"PRED: {best['street'][:25]}\nError: {row['error_m']:.0f}m | Score: {row['dino_score']:.2f}",
                                fontsize=9, color="red")
    axes[row_idx, 1].axis("off")

plt.suptitle("Top-10 Worst Predictions (Highest Error)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "v5_worst_predictions.png")
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {fig_path}")


In [ ]:
# ============================================================
# Hücre 9: PERCEPTUAL ALIASING ANALİZİ
# ============================================================
# Perceptual aliasing = farklı lokasyonlar ama çok benzer görünüm
# Bu analiz, yüksek DINO skoru AMA yanlış lokasyon olan durumları inceler.

print(f"{'='*60}")
print(f"🔄 PERCEPTUAL ALIASING ANALİZİ")
print(f"{'='*60}")

# Yüksek güvenle yanlış tahmin = Perceptual Aliasing
high_conf_wrong = df_errors[
    (~df_errors["correct_street"]) &
    (df_errors["dino_score"] > df_errors["dino_score"].quantile(0.75))  # Üst çeyrek skor
]

all_wrong = df_errors[~df_errors["correct_street"]]

print(f"\n── Perceptual Aliasing Tespiti ──")
print(f"   Toplam yanlış tahmin : {len(all_wrong)}")
print(f"   Yüksek güvenli yanlış: {len(high_conf_wrong)} "
      f"(skor > {df_errors['dino_score'].quantile(0.75):.3f})")
print(f"   Aliasing oranı       : {len(high_conf_wrong)/len(all_wrong)*100:.1f}% "
      f"(yanlışların yüksek güvenli olanı)")

# Bu vakaların ortalama hatası
if len(high_conf_wrong) > 0:
    print(f"\n   Yüksek güvenli yanlış — medyan hata: {high_conf_wrong['error_m'].median():.0f}m")
    print(f"   Düşük güvenli yanlış — medyan hata: "
          f"{all_wrong[~all_wrong.index.isin(high_conf_wrong.index)]['error_m'].median():.0f}m")

# Hangi sokaklar aliasing'e en çok maruz?
if len(high_conf_wrong) > 0:
    aliasing_streets = high_conf_wrong["query_street"].value_counts()
    print(f"\n── Aliasing'e En Çok Maruz Sokaklar ──")
    for street, count in aliasing_streets.head(10).items():
        total_queries_for_street = (df_errors["query_street"] == street).sum()
        print(f"   {street}: {count}/{total_queries_for_street} sorgu aliasing "
              f"({count/total_queries_for_street*100:.0f}%)")

# Top-K içindeki çeşitlilik analizi
print(f"\n── Top-{TOP_K} Çeşitlilik Analizi ──")
unique_streets_in_topk = []
for i in range(len(df_query)):
    streets = set()
    for k in range(TOP_K):
        streets.add(db_meta[idxs_all[i, k]]["street"])
    unique_streets_in_topk.append(len(streets))

unique_streets_arr = np.array(unique_streets_in_topk)
print(f"   Ortalama benzersiz sokak (Top-{TOP_K}): {unique_streets_arr.mean():.1f}")
print(f"   Medyan: {np.median(unique_streets_arr):.0f}")
print(f"   Sadece 1 sokak (tam tutarlılık): {(unique_streets_arr == 1).sum()} "
      f"({(unique_streets_arr == 1).mean()*100:.1f}%)")
print(f"   5+ farklı sokak (yüksek belirsizlik): {(unique_streets_arr >= 5).sum()} "
      f"({(unique_streets_arr >= 5).mean()*100:.1f}%)")

# Görselleştirme
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sol: Skor dağılımı — doğru vs yanlış
axes[0].hist(df_errors[df_errors["correct_street"]]["dino_score"], bins=40, alpha=0.6,
             color="green", label="Correct Street", density=True)
axes[0].hist(df_errors[~df_errors["correct_street"]]["dino_score"], bins=40, alpha=0.6,
             color="red", label="Wrong Street", density=True)
axes[0].set_xlabel("DINOv2 Cosine Similarity")
axes[0].set_ylabel("Density")
axes[0].set_title("Score Distribution: Correct vs Wrong")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Orta: Top-K çeşitlilik histogramı
axes[1].hist(unique_streets_arr, bins=range(1, TOP_K+2), color="#f59e0b", edgecolor="white",
             align='left', rwidth=0.8)
axes[1].set_xlabel(f"Unique Streets in Top-{TOP_K}")
axes[1].set_ylabel("Number of Queries")
axes[1].set_title("Retrieval Diversity (Perceptual Aliasing Indicator)")
axes[1].grid(alpha=0.3)

# Sağ: Aliasing vakaları harita üzerinde
if len(high_conf_wrong) > 0:
    axes[2].scatter(df_errors[df_errors["correct_street"]]["query_lon"],
                    df_errors[df_errors["correct_street"]]["query_lat"],
                    c="green", s=3, alpha=0.3, label="Correct")
    axes[2].scatter(high_conf_wrong["query_lon"], high_conf_wrong["query_lat"],
                    c="red", s=15, alpha=0.8, marker="x", label="Perceptual Aliasing")
    axes[2].set_xlabel("Longitude"); axes[2].set_ylabel("Latitude")
    axes[2].set_title("Spatial Distribution of Aliasing Cases")
    axes[2].legend(fontsize=9)
    axes[2].set_aspect('equal')
    axes[2].grid(alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "v5_perceptual_aliasing.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {fig_path}")


In [ ]:
# ============================================================
# Hücre 10: PERCEPTUAL ALIASING — GÖRSEL ÖRNEKLER
# ============================================================
# Yüksek skor ama yanlış lokasyon olan örnekleri göster

if len(high_conf_wrong) > 0:
    # En yüksek skorlu yanlış tahminler
    worst_aliasing = high_conf_wrong.nlargest(min(6, len(high_conf_wrong)), "dino_score")

    fig, axes = plt.subplots(len(worst_aliasing), 2, figsize=(12, 4*len(worst_aliasing)))
    if len(worst_aliasing) == 1:
        axes = axes.reshape(1, 2)

    for row_idx, (_, row) in enumerate(worst_aliasing.iterrows()):
        qi = row["query_idx"]

        q_img = Image.open(df_query.iloc[qi]["filepath"])
        axes[row_idx, 0].imshow(q_img)
        axes[row_idx, 0].set_title(f"QUERY: {row['query_street'][:25]}", fontsize=10, color="blue")
        axes[row_idx, 0].axis("off")

        p_img = Image.open(db_meta[idxs_all[qi, 0]]["filepath"])
        axes[row_idx, 1].imshow(p_img)
        axes[row_idx, 1].set_title(
            f"FALSE MATCH: {row['pred_street'][:25]}\n"
            f"Score: {row['dino_score']:.3f} | Error: {row['error_m']:.0f}m",
            fontsize=10, color="red")
        axes[row_idx, 1].axis("off")

    plt.suptitle("Perceptual Aliasing Examples\n(High confidence, wrong location)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig_path = os.path.join(RESULTS_DIR, "v5_aliasing_examples.png")
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✅ Kaydedildi: {fig_path}")
else:
    print("ℹ️ Yüksek güvenli yanlış tahmin bulunamadı — aliasing düşük!")

print(f"\n{'='*60}")
print(f"✅ TÜM ANALİZLER TAMAMLANDI")
print(f"   Sonuçlar: {RESULTS_DIR}")
print(f"{'='*60}")
